# Aplicações com SAM

O SAM (*Segment Anything Model*, da Meta) recebe uma imagem e um prompt, um ponto ou uma caixa, e devolve a máscara do objeto indicado, sem classe. Este notebook usa o SAM 2.1 em quatro aplicações, uma para cada forma de prompt: recorte de uma foto por clique, anonimização de pessoas com as caixas do YOLO, contagem de moedas sem prompt e rastreamento de um objeto em vídeo.

In [ ]:
import os
import subprocess
import urllib.request

import cv2
import matplotlib.pyplot as plt
import numpy as np
from IPython.display import Video
from ultralytics import SAM, YOLO
from ultralytics.models.sam import SAM2VideoPredictor

A `segment` devolve as máscaras como matrizes booleanas do tamanho da imagem, uma por objeto, e a `paint` pinta uma máscara sobre a imagem. A `save_video` grava frames em um MP4 que o navegador consegue tocar.

In [ ]:
sam = SAM("sam2.1_b.pt")   # 154 MB, baixa na primeira vez


def download(url, name):
    path = os.path.join("data", name)
    if not os.path.exists(path):
        os.makedirs("data", exist_ok=True)
        urllib.request.urlretrieve(url, path)
    return path


# Devolve um array bool [N, H, W], uma máscara por objeto
def segment(image, **prompt):
    return sam(image, verbose=False, **prompt)[0].masks.data.cpu().numpy().astype(bool)


def show(*images, width=10):
    fig, axes = plt.subplots(1, len(images), figsize=(width * len(images), width * images[0].shape[0] / images[0].shape[1]))
    for ax, image in zip(np.atleast_1d(axes), images):
        ax.imshow(image[:, :, ::-1])   # OpenCV usa BGR, o matplotlib espera RGB
        ax.axis("off")
    plt.show()


def paint(image, mask, color=(0, 0, 255)):
    image = image.copy()
    image[mask] = (0.5 * image[mask] + 0.5 * np.array(color)).astype(np.uint8)
    return image


def ffmpeg(*args):
    subprocess.run(["ffmpeg", "-y", "-loglevel", "error", *args], check=True)


def save_video(frames, path, fps=25):
    writer = cv2.VideoWriter("data/temp.mp4", cv2.VideoWriter_fourcc(*"mp4v"), fps, (1280, 720))
    for frame in frames:
        writer.write(frame)
    writer.release()
    ffmpeg("-i", "data/temp.mp4", "-c:v", "h264", "-pix_fmt", "yuv420p", path)   # o navegador só toca H.264
    return Video(path, embed=True, width=800)

## Recorte por clique

O prompt é um ponto $(x, y)$ sobre a pessoa, com rótulo `1`, que indica que o ponto pertence ao objeto.

In [ ]:
photo = cv2.imread(download("https://ultralytics.com/images/zidane.jpg", "zidane.jpg"))
show(photo)

Com a máscara, o efeito retrato dos celulares consiste em escolher, pixel a pixel, entre a foto e uma versão borrada dela.

In [ ]:
point = [300, 400]   # [x, y] em pixels
mask = segment(photo, points=[point], labels=[1])[0]   # [H, W]. Rótulo 1 quer dizer que o ponto é do objeto

# Dentro da máscara fica a foto, fora fica a versão borrada
portrait = np.where(mask[:, :, None], photo, cv2.GaussianBlur(photo, (51, 51), 0))

marked = paint(photo, mask)
cv2.circle(marked, point, 10, (0, 255, 255), -1)
show(marked, portrait, width=7)

Um ponto na gravata pode indicar a gravata ou a pessoa. Antes de rodar, tente prever o que o SAM devolve. Um segundo ponto, no rosto, desfaz a ambiguidade: os pontos de um mesmo objeto vão juntos em uma lista.

In [ ]:
tie, face = [470, 550], [600, 330]
one_point = segment(photo, points=[tie], labels=[1])[0]

# Os dois pontos juntos numa lista indicam um objeto só: points=[[ponto1, ponto2]], labels=[[1, 1]]
two_points = segment(photo, points=[[tie, face]], labels=[[1, 1]])[0]
show(paint(photo, one_point), paint(photo, two_points), width=7)

## Anonimização

O YOLO encontra as pessoas, e as caixas dele viram os prompts do SAM.

In [ ]:
street = cv2.imread(download("https://ultralytics.com/images/bus.jpg", "bus.jpg"))
show(street, width=6)

À esquerda, a solução sem o SAM, que borra a caixa inteira e leva junto o ônibus atrás das pessoas. À direita, apenas as silhuetas borradas.

In [ ]:
detector = YOLO("yolo26n.pt")
# Lista de caixas [[x0, y0, x1, y1], ...]. Classe 0 é pessoa
boxes = detector.predict(street, classes=[0], verbose=False)[0].boxes.xyxy.int().tolist()

# Cada caixa do YOLO vira um prompt. Juntando as máscaras, temos todas as pessoas
people = segment(street, bboxes=boxes).any(axis=0)   # [N, H, W] vira [H, W]
print(f"{len(boxes)} pessoas detectadas")

blurred = cv2.GaussianBlur(street, (99, 99), 0)

# Sem o SAM, borra o retângulo inteiro
box_blur = street.copy()
for x0, y0, x1, y1 in boxes:
    box_blur[y0:y1, x0:x1] = blurred[y0:y1, x0:x1]

# Com o SAM, borra só as pessoas
show(box_blur, np.where(people[:, :, None], blurred, street), width=7)

A pessoa cortada na borda direita fica sem borrar, porque o YOLO não a detectou. O SAM segmenta apenas o que recebe como prompt, então a cobertura da anonimização é a do detector.

## Contagem de objetos

Sem prompt, o SAM espalha uma grade de pontos pela imagem e devolve uma máscara para cada objeto que encontra. A foto tem 24 moedas.

In [ ]:
coins = cv2.imread(download("https://raw.githubusercontent.com/scikit-image/scikit-image/v0.19.3/skimage/data/coins.png", "coins.png"))
show(coins, width=6)

Antes de rodar, tente prever quantas máscaras o SAM devolve.

In [ ]:
masks = segment(coins)   # [N, H, W], uma máscara por objeto encontrado
print(f"{len(masks)} máscaras")

colored = coins.copy()
for mask in masks:
    colored = paint(colored, mask, np.random.randint(0, 255, 3))
show(colored, width=6)

Além das moedas, o SAM devolve o fundo e, para algumas moedas, uma segunda máscara com só uma parte dela. A lógica de aplicação descarta as máscaras com mais de 10% da imagem e, da maior para a menor, as que têm mais de 80% da área dentro de uma já aceita. Um limite de área mínima falharia, porque algumas dessas partes são maiores que as moedas menores. O diâmetro de cada moeda sai da área, $d = 2\sqrt{A / \pi}$, em pixels.

In [ ]:
coin_masks = []   # lista de máscaras [H, W] aceitas como moeda
for mask in sorted(masks, key=lambda m: -m.sum()):   # da maior para a menor
    area = mask.sum()
    is_background = area >= 0.1 * mask.size
    # Se 80% da máscara cai dentro de uma moeda já aceita, é um pedaço dela
    is_part = any((mask & other).sum() / area >= 0.8 for other in coin_masks)
    if not is_background and not is_part:
        coin_masks.append(mask)
print(f"{len(coin_masks)} moedas")

labeled = coins.copy()
for mask in coin_masks:
    labeled = paint(labeled, mask, (0, 200, 0))
    ys, xs = np.nonzero(mask)                       # pixels da moeda, para achar o centro
    diameter = 2 * np.sqrt(mask.sum() / np.pi)      # diâmetro de um círculo com a mesma área
    cv2.putText(labeled, f"{diameter:.0f}", (int(xs.mean()) - 8, int(ys.mean()) + 4), cv2.FONT_HERSHEY_SIMPLEX, 0.4, (255, 255, 255), 1)
show(labeled, width=6)

## Rastreamento em vídeo

O SAM 2 guarda uma memória das máscaras dos frames anteriores e, com ela, segue pelo vídeo o objeto indicado no primeiro frame. Usamos os primeiros 6 segundos de um vídeo de esqui, reduzidos para 1280×720.

In [ ]:
skiing = download("https://media.roboflow.com/supervision/video-examples/skiing.mp4", "skiing.mp4")
ffmpeg("-i", skiing, "-t", "6", "-vf", "scale=1280:720", "-c:v", "h264", "-pix_fmt", "yuv420p", "data/skiing-short.mp4")
Video("data/skiing-short.mp4", embed=True, width=800)

O prompt é um ponto sobre o esquiador no primeiro frame. A aplicação é o efeito de cor seletiva: o esquiador em cores e o resto em tons de cinza.

In [ ]:
skier = [715, 420]
tracker = SAM2VideoPredictor(overrides=dict(model="sam2.1_b.pt", imgsz=1024, save=False, verbose=False))


def color_splash(result):
    frame = result.orig_img
    gray = cv2.cvtColor(cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY), cv2.COLOR_GRAY2BGR)   # cinza com 3 canais
    mask = result.masks.data[0].cpu().numpy().astype(bool)   # masks.data é [1, H, W], um objeto só
    # Esquiador colorido, resto em cinza
    return np.where(mask[:, :, None], frame, gray)


# O ponto marca o esquiador no primeiro frame. Depois o SAM 2 segue ele sozinho
results = tracker(source="data/skiing-short.mp4", points=[skier], labels=[1], stream=True)
save_video((color_splash(result) for result in results), "data/skiing-splash.mp4")

O esquiador é seguido em todo o trecho, com mudanças de tamanho e de pose, e os outros esquiadores ficam fora da máscara. Em uma RTX 3050 de notebook, cada frame leva cerca de 1 s. Para vídeos longos, a alternativa é a da anonimização: um detector rápido em cada frame e o SAM apenas onde a forma importa.